In [ ]:
import sys
from pathlib import Path
import pandas as pd
import seaborn as sns
import numpy as np

modules_path = Path.cwd().parent / "modules"
sys.path.append(str(modules_path))
from rdkit import Chem
from rdkit.Chem import DataStructs

from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem import DataStructs

import numpy as np
import matplotlib.pyplot as plt
import os
import datasplits
from HQ_featureextractor import chemdescriptor

import re
from __future__ import annotations
from dataclasses import dataclass
from typing import Dict, Any, Union
from pathlib import Path

In [ ]:
# Convert SMILES to RDKit molecules
def mol_from_smiles(smiles_list):
    return [Chem.MolFromSmiles(s) for s in smiles_list]

# Generate Morgan fingerprints using modern RDKit API
def fingerprints(mols):
    fmgen = rdFingerprintGenerator.GetMorganGenerator(radius=2)
    return [fmgen.GetFingerprint(m) for m in mols]

In [ ]:
# Import data
app2024 = pd.read_csv("../01_DATASETS/APP2024_original.csv")

In [ ]:
app2024["prop"].unique()

In [ ]:
def clean_smiles(smiles_list):
    cleaned_x_idxs = []
    for idx, s in enumerate(smiles_list):
        if '.' in s:
            continue
        cleaned_x_idxs.append(idx) 
    return cleaned_x_idxs

def inchikey(smiles):
    m = Chem.MolFromSmiles(smiles)
    return Chem.inchi.MolToInchiKey(m) if m else None

In [ ]:
cleaned_idx = clean_smiles(app2024["SMILES"]) # remove salts and co-crystals 
app2024_desalt = app2024.iloc[cleaned_idx]
app2024_desalt.groupby("prop")["val"].agg(["count", "mean", "std", "max", "min"])

In [ ]:
cd = app2024_desalt[app2024_desalt["prop"] == "CD"]

In [ ]:
## Check overlap with pairwise tanimoto coefs
mols = mol_from_smiles(cd["SMILES"])
fps = fingerprints(mols)

In [ ]:
matches = []
for idx, fp in enumerate(fps):
    for idx2, fp2 in enumerate(fps):
        if idx != idx2:
            tc = DataStructs.TanimotoSimilarity(fp, fp2)
            if tc == 1:
                matches.append((idx, idx2))

### Crystal Density
- because the same molecule has multiple other targets, they have been repeated in the dataset. They should have the same SMILES + cd though 

In [ ]:
app_cd = app2024_desalt[["SMILES", "OB", "Density [g/cc]" ,"X"]].copy()
app_cd.columns = ["SMILES", "OB", "cd", "X"]
app_dedup = app_cd.drop_duplicates(subset="SMILES").copy()

app_dedup.rename(columns={"cd": "val"}, inplace=True)
app_dedup.drop(175, inplace=True)
#app_dedup.to_csv("../01_DATASETS/CD/CD.csv", index=False)

In [ ]:
data = {
    'HOF exp': app2024_desalt[app2024_desalt["prop"] == "HOF exp"]["val"].values,
    'log(h50) exp': app2024_desalt[app2024_desalt["prop"] == "log(h50) exp"]["val"].values,
    "CD": app_dedup["val"].values,
}
df = pd.concat([pd.DataFrame({k: v}) for k, v in data.items()], axis=1)
df_melted = df.melt(var_name='property', value_name='value')

## Include QU dataset at IS
* b-HMX had a range, i looked at one of the papers which had it at 32 cm as a reference so I fixed a value of 7.85 J
* censored values were dropped

In [ ]:
QU_data = pd.read_csv("../01_DATASETS/QU_IS.csv")

# drop any values in Exp IS (J) that are not numeric
QU_data = QU_data[pd.to_numeric(QU_data["Exp IS (J)"], errors='coerce').notnull()]
QU_df = QU_data[["Name", "SMILES", "Exp IS (J)", "Compound Type"]].copy()
QU_df.rename(columns={
    "Name": "name",
    "SMILES": "SMILES",
    "Exp IS (J)": "val",
    "Compound Type": "type"
}, inplace=True)
QU_df.reset_index(drop=True)
QU_df["val"] = QU_df["val"].astype("float32")
QU_df["source"] = "QUAYLE"
#QU_df.to_csv("../01_DATASETS/QU_IS/QU_IS.csv", index=False)

In [ ]:
sns.histplot(data=QU_df, x="val")

In [ ]:
from scipy.constants import g
APP_IS = app2024_desalt[app2024_desalt["prop"] == "log(h50) exp"].copy()
APP_IS["IS_m"] = np.power(10, APP_IS["val"]) * 0.01 # cm -> m
APP_IS["IS_J"] = 2.5 * g * APP_IS["IS_m"]
APP_IS["source"] = "MATHIEU"

In [ ]:
sns.histplot(data=APP_IS, x="IS_J")

## mix IS datasets

In [ ]:
manner_data = pd.read_csv("../01_DATASETS/MANNER_IS.csv")
select_relevant_columns = ["SMILES_dat", "logImpact_J"]

# canonicalize smiles
mols = [Chem.MolFromSmiles(s) for s in manner_data["SMILES_dat"]]
manner_data["SMILES"] = [Chem.MolToSmiles(m, canonical=True) for m in mols]

# unlog IS values (J) -- assume log_10?
manner_data["Impact_J"] = 10 ** manner_data["logImpact_J"]

manner = manner_data[["SMILES", "Impact_J"]].copy()
manner.rename(columns={"Impact_J": "val"}, inplace=True)
manner["source"] = "MARRS"

In [ ]:
QU_IS = QU_df[["SMILES", "val", "source"]].copy()
APP_IS = APP_IS[["SMILES", "val", "source"]].copy()

IS_df = pd.concat([QU_IS, APP_IS, manner]).drop_duplicates(subset="SMILES", keep="first").reset_index(drop=True)
IS_df["val"] = IS_df["val"].astype("float32")

mols = mol_from_smiles(IS_df["SMILES"].values)
IS_df["canon_SMILES"] = [Chem.MolToSmiles(m, canonical=True) for m in mols]

orig_len = len(IS_df)
IS_df = IS_df.drop_duplicates(subset="canon_SMILES", keep="first").reset_index(drop=True)
IS_df.drop(index=528, inplace=True)
IS_df["IS (J)"] = IS_df["val"]
IS_df["val"] = np.log(IS_df["IS (J)"])
print(f"Dropped {orig_len - len(IS_df)} molecules")

sns.histplot(data=IS_df, x="val", bins=20)
plt.xlabel("IS (J)")

In [ ]:
len(IS_df)

In [ ]:
IS_df.rename(columns={"source": "Source"}, inplace=True)
sns.histplot(data=IS_df, x="val", bins=30, hue="Source")
plt.xlabel("IS (logJ)")
plt.show()

## load datasets

In [ ]:
def load_csv_by_subdir(datasets_dir: str | Path) -> dict[str, object]:
    datasets_dir = Path(datasets_dir).resolve()
    data_dict: dict[str, object] = {}

    for subdir in sorted(p for p in datasets_dir.iterdir() if p.is_dir()):
        csv_files = sorted(subdir.glob("*.csv"))

        data_dict[subdir.name] = pd.read_csv(csv_files[0])

    return data_dict

# example
targets = load_csv_by_subdir("../01_DATASETS/")

In [ ]:
dataframes = {}
for name, data in targets.items():
    if name == "CD":
        name = r"$\rho_{cryst}$"

    data["Dataset"] = name
    dataframes[name] = data[["Dataset", "SMILES", "val"]]

merged = pd.concat(dataframes.values(), ignore_index=True)

# Publication-style defaults
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)

datasets = merged["Dataset"].unique()

g = sns.catplot(
    data=merged,
    col="Dataset",
    col_order=datasets,
    y="val",
    kind="violin",
    sharey=False,
    height=3,
    aspect=1,
    cut=0,
    inner="box",
    linewidth=0.8,
    color=sns.color_palette("tab10")[0],
)

# Style each subplot
for ax, ds in zip(g.axes.flat, datasets):
    ax.set_title(f"{ds}", fontsize=12)
    ax.set_xlabel("Count")
    ax.grid(True, axis="y", linestyle="--", linewidth=0.5, alpha=0.7)

    # Clean spines
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # Dataset-specific y-labels
    if ds == datasets[0]:
        ax.set_ylabel("kJ/mol")
    elif ds == datasets[1]:
        ax.set_ylabel("g/cm$^3$")
    else:
        ax.set_ylabel("log(J)")

# Publication sizing / spacing
g.figure.set_dpi(300)
g.figure.set_size_inches(14, 4)
plt.tight_layout()
plt.show()

## Data Splitting
* motivation: create 3 splitting methods with 3 different levls of test-train distribution shifts 
1. Random split (in distribution)

In [ ]:
splits = {}
for target, data in targets.items():
    base_dir = f"../01_DATASETS/{target}/"
    print(f"Processing target: {target}")
    
    # Generate random splits
    for random_state in [0, 1, 2, 3, 4]:
        _, rand_test = datasplits.random_split_indices(len(data), test_size=0.2, random_state=random_state)    
        np.save(os.path.join(base_dir, f"random_seed{random_state}_split_test.npy"), rand_test)

### Use grouping in fingerprint space to generate splittings 

In [ ]:
from __future__ import annotations
from typing import Iterable, List, Sequence, Tuple, Optional
from rdkit.Chem import rdFingerprintGenerator
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from sklearn.metrics import silhouette_score

# Convert SMILES to RDKit molecules
def mol_from_smiles(smiles_list):
    return [Chem.MolFromSmiles(s) for s in smiles_list]

# Generate Morgan fingerprints using modern RDKit API
def fingerprints(mols):
    fmgen = rdFingerprintGenerator.GetMorganGenerator(radius=2)
    return [fmgen.GetFingerprint(m) for m in mols]
    
def reduce_dimensions(
    X: np.ndarray,
    n_components: int = 100,
    random_state: int = 1,
) -> Tuple[np.ndarray, TruncatedSVD]:
    """
    Reduce fingerprint matrix dimensionality with TruncatedSVD.

    Parameters
    ----------
    X : np.ndarray
        Input matrix (n_samples, n_features).
    n_components : int, default=100
        Number of SVD components.
    random_state : int, default=42
        Random state for reproducibility.

    Returns
    -------
    X_reduced : np.ndarray
        Reduced embeddings (n_samples, min(n_components, n_features-1)).
    svd : TruncatedSVD
        Fitted TruncatedSVD model.
    """
    
    if X.ndim != 2:
        raise ValueError(f"X must be 2D, got shape {X.shape}")

    if X.shape[0] == 0:
        svd = TruncatedSVD(n_components=min(n_components, max(1, X.shape[1])), random_state=random_state)
        return np.empty((0, 0), dtype=np.float32), svd

    # TruncatedSVD requires n_components <= n_features - 1 for meaningful reduction.
    max_components = max(1, min(n_components, X.shape[1] - 1, X.shape[0] - 1))
    svd = TruncatedSVD(n_components=max_components, random_state=random_state)
    X_reduced = svd.fit_transform(X)
    return X_reduced.astype(np.float32, copy=False), svd


def cluster_embeddings(
    X_emb: np.ndarray,
    n_clusters: int = 5,
    random_state: int = 42,
    batch_size: int = 1024,
    n_init: str | int = "auto",
) -> Tuple[np.ndarray, MiniBatchKMeans]:
    """
    Cluster embeddings with MiniBatchKMeans.

    Parameters
    ----------
    X_emb : np.ndarray
        Embedding matrix (n_samples, n_dims).
    n_clusters : int, default=5
        Number of clusters.
    random_state : int, default=42
        Random seed.
    batch_size : int, default=1024
        Mini-batch size.
    n_init : str | int, default="auto"
        Number of initializations.

    Returns
    -------
    labels : np.ndarray
        Cluster labels (n_samples,).
    model : MiniBatchKMeans
        Fitted clustering model.
    """
    if X_emb.ndim != 2:
        raise ValueError(f"X_emb must be 2D, got shape {X_emb.shape}")
    elif X_emb.shape[0] < n_clusters:
        raise ValueError(
            f"Need at least n_clusters samples. Got {X_emb.shape[0]} samples for n_clusters={n_clusters}."
        )
    else:
        model = MiniBatchKMeans(
            n_clusters=n_clusters,
            random_state=random_state,
            batch_size=batch_size,
            n_init=n_init,
        )
        labels = model.fit_predict(X_emb)
        return labels.astype(int, copy=False), model


def build_output(valid_smiles: Sequence[str], labels: np.ndarray) -> pd.DataFrame:
    """
    Build output DataFrame.

    Parameters
    ----------
    valid_smiles : Sequence[str]
        Valid SMILES kept after parsing.
    labels : np.ndarray
        Cluster labels for each valid SMILES.

    Returns
    -------
    df : pd.DataFrame
        DataFrame with columns: original_smiles, cluster_id
    """
    if len(valid_smiles) != len(labels):
        raise ValueError(f"Length mismatch: {len(valid_smiles)=} vs {len(labels)=}")

    df = pd.DataFrame(
        {
            "original_smiles": list(valid_smiles),
            "cluster_id": labels,
        }
    )
    return df


def cluster_smiles_into_5(
    smiles_list: Sequence[str],
    *,
    use_counts: bool = False,
) -> pd.DataFrame:
    """
    Cluster SMILES into exactly 5 clusters using:
    RDKit parsing -> Morgan fingerprints (ECFP4) -> TruncatedSVD -> MiniBatchKMeans.

    Parameters
    ----------
    smiles_list : Sequence[str]
        Input SMILES.
    use_counts : bool, default=False
        If True, attempt count fingerprints; otherwise use bit vectors.

    Returns
    -------
    pd.DataFrame
        Columns: original_smiles, cluster_id (only for valid SMILES).
    """
    mols = mol_from_smiles(smiles_list)
    X_fp = np.array(fingerprints(mols))

    X_emb, svd = reduce_dimensions(X_fp, n_components=200, random_state=42)
    X_emb = normalize(X_emb)

    # SVD explained variance ratio can contain NaNs if degenerate; make printing robust.
    evr = getattr(svd, "explained_variance_ratio_", None)
    if evr is not None and len(evr) > 0:
        evr_sum = float(np.nansum(evr))
        print(f"TruncatedSVD explained variance ratio (sum over components): {evr_sum:.4f}")
    else:
        print("TruncatedSVD explained variance ratio: unavailable")

    labels, _ = cluster_embeddings(
        X_emb,
        n_clusters=5,
        random_state=42,
        batch_size=1024,
        n_init=50,
    )

    sil = silhouette_score(X_emb, labels)
    print(f"Silhouette score: {sil}")

    df = build_output(smiles_list, labels)

    # Molecules per cluster
    counts = df["cluster_id"].value_counts().sort_index()
    print("Molecules per cluster:")
    for cid, cnt in counts.items():
        print(f"  Cluster {cid}: {cnt}")

    return df


In [ ]:
base_dir = Path.cwd().parent.joinpath("01_DATASETS")

for dataset_name, dataset_df in targets.items():
    # generate clusters
    smiles = dataset_df["SMILES"].values
    cluster_df = cluster_smiles_into_5(smiles)
    dataset_df["cluster_id"] = cluster_df["cluster_id"]

    # generate splits based on clusters
    for label in dataset_df["cluster_id"].unique():
        cluster = dataset_df[dataset_df["cluster_id"] == label]
        idxs = cluster.index
        filepath = base_dir.joinpath(f"{dataset_name}").joinpath(f"cluster_label{label}_split_test.npy")
        np.save(filepath, idxs)